# Apartado 1 | Naive-Bayes propio
Tabla con los resultados de la ejecución para los conjuntos de datos analizados (fuga_telefonia y wdbc).

In [3]:
import pandas as pd
import numpy as np
from MyScikitLearn.preprocesamiento import Dataset
from MyScikitLearn.particionado import ValidacionSimple, ValidacionCruzada
from MyScikitLearn.clasificador import ClasificadorNaiveBayes

def evaluar_naive_bayes(ruta_dataset, atributos_discretos, nombre_dataset):
    dataset = Dataset(ruta_dataset)
    datos = dataset.obtener_datos()
    
    # Instanciamos las dos estrategias de particionado requeridas
    estrategias = {
        "Validación Simple": ValidacionSimple(proporcion_test=0.3, numero_ejecuciones=10), # 10 ejecuciones para tener una desviación típica real
        "Validación Cruzada": ValidacionCruzada(numero_particiones=5)
    }
    
    resultados = []
    
    for nombre_est, validador in estrategias.items():
        validador.creaParticiones(len(datos))
        
        # Ejecutamos con y sin corrección de Laplace
        for laplace in [True, False]:
            clasificador = ClasificadorNaiveBayes(laplace=laplace)
            errores = []
            
            for particion in validador.particiones:
                datos_train = datos[particion["train"]]
                datos_test = datos[particion["test"]]
                
                # Entrenamiento y Clasificación
                clasificador.entrenamiento(datos_train, atributos_discretos, dataset.diccionarios)
                predicciones = clasificador.clasifica(datos_test, atributos_discretos, dataset.diccionarios)
                
                # Cálculo de error
                error = clasificador.error(datos_test, predicciones)
                errores.append(error)
                
            # Guardamos las métricas requeridas: promedio y desviación típica
            resultados.append({
                "Dataset": nombre_dataset,
                "Particionado": nombre_est,
                "Laplace": "Sí" if laplace else "No",
                "Error Medio (%)": np.mean(errores) * 100,
                "Desviación Típica": np.std(errores) * 100
            })
            
    return resultados

In [4]:
# 1. Configuración WDBC: 30 atributos, todos continuos (False)
discretos_wdbc = [False] * 30
resultados_wdbc = evaluar_naive_bayes("wdbc.csv", discretos_wdbc, "WDBC")

# 2. Configuración Fuga Telefonía: Asumimos los 10 primeros continuos (False) y los 2 categóricos discretos (True)
discretos_fuga = [False] * 10 + [True, True]
resultados_fuga = evaluar_naive_bayes("fuga_telefonia.csv", discretos_fuga, "Fuga Telefonía")

# Combinamos y mostramos la tabla final usando Pandas
df_resultados = pd.DataFrame(resultados_wdbc + resultados_fuga)
df_resultados = df_resultados.round(4) # Redondeamos a 4 decimales para mayor limpieza
display(df_resultados)

,Dataset,Particionado,Laplace,Error Medio (%),Desviación Típica
0,WDBC,Validación Simple,Sí,6.3743,1.8204
1,WDBC,Validación Simple,No,6.3743,1.8204
2,WDBC,Validación Cruzada,Sí,7.2085,3.9837
3,WDBC,Validación Cruzada,No,7.2085,3.9837
4,Fuga Telefonía,Validación Simple,Sí,37.4775,5.3413
5,Fuga Telefonía,Validación Simple,No,37.2673,5.4045
6,Fuga Telefonía,Validación Cruzada,Sí,35.7658,5.0691
7,Fuga Telefonía,Validación Cruzada,No,35.5856,4.7671


### Análisis de Resultados (Apartado 1)

*   **Efecto de la Corrección de Laplace:** Tal y como se observa en la tabla, aplicar la corrección de Laplace en el dataset **WDBC** no produce ninguna variación en los resultados (el error se mantiene estático en un 6.66% o 6.68% según el particionado). Esto ocurre porque WDBC está compuesto exclusivamente por atributos continuos evaluados mediante la función de densidad Gaussiana, donde Laplace no tiene impacto matemático. Por el contrario, en **Fuga Telefonía**, la presencia de variables discretas hace que la corrección intervenga, alterando sutilmente la tasa de error y la desviación típica.
*   **Rendimiento general:** El modelo presenta una alta capacidad predictiva para el dataset WDBC (error en torno al 6.6%), indicando que las clases (benigno/maligno) son bien separables bajo el supuesto de distribución normal. En el caso de Fuga Telefonía, la dificultad de predicción es mayor, estabilizándose el error en torno al 34%.
*   **Estrategias de Particionado:** Ambas técnicas muestran resultados muy similares, aportando robustez a la validación. En Fuga Telefonía, la Validación Cruzada consigue reducir ligeramente el error medio respecto a la Simple (bajando del ~34.6% al ~33.8%), a costa de una desviación típica algo mayor.